In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Sistema creado con antigua base de datos
# Está deprecada por nuevos campos 
# Sub-Categoría, Periodicidad
#
#
#


# ----- SISTEMA DE VISUALIZACIÓN DE AHORRO ----- #

# 1° Convertir CSV en objetos Pandas
# Creamos un Dataframe
uri = "Data/GastosAgo25-Feb26.csv"
df = pd.read_csv(
    uri,
    encoding='utf-8',
    usecols=[0, 1, 2, 3, 4], 
    skipinitialspace=True
)



# ----- LIMPIEZA ----- #

# Nos aseguramos que los datos tengan un tipo de datos que nos
# permita manipularlos

# MONTO COMO NUMERO #
# Elimina símbolos de moneda, puntos de miles y convierte a número
# Ajusta el replace según cómo se vea tu moneda (ej: '$', '.', ',')
df['Monto'] = df['Monto'].replace('[\$,]', '', regex=True).astype(float)

# FECHA COMO DATETIME #
# Diccionario para traducir meses
meses_esp = {
    'enero': 'January', 'febrero': 'February', 'marzo': 'March',
    'abril': 'April', 'mayo': 'May', 'junio': 'June',
    'julio': 'July', 'agosto': 'August', 'septiembre': 'September',
    'octubre': 'October', 'noviembre': 'November', 'diciembre': 'December'
}

# 1. Pasamos a minúsculas y quitamos la palabra "de"
# "1 de agosto de 2025" -> "1 agosto 2025"
df['Fecha_Limpia'] = df['Fecha'].str.lower().str.replace(' de ', ' ', regex=False)

# 2. Traducimos los meses usando nuestro diccionario
for esp, ing in meses_esp.items():
    df['Fecha_Limpia'] = df['Fecha_Limpia'].str.replace(esp, ing, regex=False)

# 3. Convertimos a datetime
# El formato ahora es "día mes año" -> "%d %B %Y"
df['Fecha'] = pd.to_datetime(df['Fecha_Limpia'], format='%d %B %Y')

# Borramos la columna temporal
df = df.drop(columns=['Fecha_Limpia'])



# ----- INGRESOS Y EGRESOS  ----- #
# Creamos dos mascaras de pandas para separar los que ingresa de lo que egresa
ingresos = df[df['Tipo'] == 'Ingreso'].copy()
egresos = df[df['Tipo'] == 'Egreso'].copy()

# Sacamos el promedio de gasto mensual
gasto_mensual_promedio = egresos.groupby(egresos['Fecha'].dt.to_period('M'))['Monto'].sum().mean()

print(f"Tu gasto mensual promedio es: ${gasto_mensual_promedio:,.2F}")



# ----- RUNWAY / TIEMPO DE VIDA ----- #
ahorro_actual = float(input("¿Cuál es tu ahorro total disponible hoy? "))

meses_supervivencia = ahorro_actual / gasto_mensual_promedio

print(f"⚠️ Con tu ritmo actual, puedes vivir {meses_supervivencia:.1f} meses.")


# Crear datos de proyección
meses = range(0, int(meses_supervivencia) + 2)
saldo_proyectado = [ahorro_actual - (gasto_mensual_promedio * m) for m in meses]

plt.figure(figsize=(10, 6))
plt.plot(meses, saldo_proyectado, marker='o', color='blue', linestyle='-')
plt.axhline(0, color='red', linewidth=1) # Línea de "bancarrota"

plt.title("Proyección de Agotamiento de Ahorros")
plt.xlabel("Meses desde hoy")
plt.ylabel("Saldo proyectado ($)")
plt.grid(False, alpha=0.3)
plt.show()

KeyError: 'Tipo'